In [1]:
"""
Computational Psycholinguistics – RQ1 Analysis Pipeline
Hindi-English Cross-Lingual Word Association Study
Tests 1-6 for RQ1: Do human word associations differ across translation-equivalent
Hindi-English cue pairs, and if so, along what semantic and cultural dimensions?

INPUT: data/responses.csv  (see DATA_FORMAT.md for column spec)
OUTPUT: results/ directory with one CSV + console summary per test

Run in Kaggle:  !python rq1_pipeline.py
Dependencies:   pandas, scipy, numpy, matplotlib, seaborn  (all pre-installed on Kaggle)
"""

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import chi2_contingency, fisher_exact, binomtest
from scipy.spatial.distance import jaccard
from itertools import combinations
from collections import Counter

warnings.filterwarnings("ignore")
os.makedirs("results", exist_ok=True)
os.makedirs("results/figures", exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

# The 10 translation pairs: (English_cue, Hindi_cue)
PAIRS = [
    ("Glasses",      "Chashma"),
    ("Dog",          "Kutta"),
    ("Cloud",        "Baadal"),
    ("Salt",         "Namak"),
    ("Time",         "Samay"),
    ("Shyness",      "Sharam"),
    ("Blessing",     "Aashirwad"),
    ("Neighbourhood","Mohalla"),
    ("Values",       "Sanskar"),
    ("Opportunity",  "Mauka"),
]

# Map every cue to its semantic category
CATEGORY_MAP = {
    "Glasses": "concrete",   "Chashma":      "concrete",
    "Dog":     "concrete",   "Kutta":        "concrete",
    "Cloud":   "concrete",   "Baadal":       "concrete",
    "Salt":    "concrete",   "Namak":        "concrete",
    "Time":    "abstract",   "Samay":        "abstract",
    "Shyness": "emotional",  "Sharam":       "emotional",
    "Blessing":"emotional",  "Aashirwad":    "cultural",
    "Neighbourhood":"concrete","Mohalla":    "cultural",
    "Values":  "abstract",   "Sanskar":      "cultural",
    "Opportunity":"abstract","Mauka":        "cultural",
}

# ─────────────────────────────────────────────────────────────────────────────
# HINDI LEXICON  (transliterated Hindi words that appear in the response set)
# Extend this list if you add participants or words.
# Rule: include only words that are NOT standard English dictionary words.
# Borderline loanwords (specs, colony) are treated as English.
# ─────────────────────────────────────────────────────────────────────────────
HINDI_WORDS = {
    # cue words
    "chashma","baadal","samay","namak","mohalla","kutta","aashirwad",
    "sharam","sanskar","mauka",
    # responses – body / physical
    "kaanch","baarish","barish","pani","paani","aasman","bijli","chawn",
    "baarish","megha","varsha","mirch","chini","shakkar","namkeen","swaad","swad",
    # responses – action / verb forms
    "bhaunkna","garajna","pehenna","pehna","khana","gavana","bhaukna",
    # responses – abstract / descriptive
    "jhijak","haya","lajja","laja","lihaj","besharmi","halal","namakharam",
    "parampara","aadarsh","acha","shubh","balwan","vartman","sunhera","kimmat",
    "kimat","vehem","befakuf",
    # responses – relational / people
    "aata","billi","chauka","raina","patti","ghadi","nazar","gali","ghar",
    "dost","sheher","padosi","gharana","parivar","parivaar","waqt","haath","muh",
    "nani","dadi","kutti","aashram","kripa","parast","sushil","sanskari",
    # proper nouns that are Hindi-origin
    "mahabharat","starplus",
    # Devanagari (if participants typed in script)
    "संस्कृति","आँखें",
}

CUE_LANG_MAP = {w: "hindi" for _, w in PAIRS}
CUE_LANG_MAP.update({w: "english" for w, _ in PAIRS})


def classify_language(word: str) -> str:
    """
    Classify a response word as 'hindi', 'english', or 'mixed'.
    Logic:
      1. Devanagari script → hindi
      2. Word (lowercased, stripped) in HINDI_WORDS → hindi
      3. Multi-word response containing both → mixed
      4. Otherwise → english
    """
    if not isinstance(word, str) or word.strip() == "":
        return "unknown"
    w = word.strip()
    # Devanagari unicode block: U+0900–U+097F
    if any("\u0900" <= c <= "\u097F" for c in w):
        return "hindi"
    tokens = w.lower().replace("-", " ").split()
    if len(tokens) > 1:
        langs = {classify_language(t) for t in tokens}
        if "hindi" in langs and "english" in langs:
            return "mixed"
        if langs == {"hindi"}:
            return "hindi"
    return "hindi" if tokens[0] in HINDI_WORDS else "english"


def load_data(path: str = "/kaggle/input/datasets/navshri2558/responses/responses.csv") -> pd.DataFrame:
    """Load and validate the responses CSV."""
    df = pd.read_csv(path)
    required = {"participant_id", "cue", "response",
                "response_type", "semantic_category", "affective_valence"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"CSV is missing columns: {missing}")

    df["cue_language"]      = df["cue"].map(CUE_LANG_MAP)
    df["response_language"] = df["response"].apply(classify_language)
    df["cue_category"]      = df["cue"].map(CATEGORY_MAP)
    pair_map = {eng: eng for eng, _ in PAIRS}
    pair_map.update({hin: eng for eng, hin in PAIRS})
    df["pair"]              = df["cue"].map(pair_map)

    print(f"✓ Loaded {len(df)} responses from {df['participant_id'].nunique()} "
          f"participants, {df['cue'].nunique()} cue words.")
    print(f"  Cue language breakdown:\n{df['cue_language'].value_counts().to_string()}\n")
    return df


# ─────────────────────────────────────────────────────────────────────────────
# HELPER: pretty chi-square / Fisher wrapper
# ─────────────────────────────────────────────────────────────────────────────

def run_chi2_or_fisher(ct: pd.DataFrame, label: str) -> dict:
    """
    Given a contingency table, run chi-square if all expected counts ≥ 5,
    otherwise run Fisher's exact (2×2 only) or report with warning.
    Returns a dict with test name, statistic, p-value, and expected counts.
    """
    arr = ct.values.astype(float)
    if arr.shape == (2, 2):
        chi2, p_chi, dof, expected = chi2_contingency(arr, correction=False)
        if (expected < 5).any():
            _, p_fisher = fisher_exact(arr)
            return {"test": "Fisher's exact", "stat": None, "p": p_fisher,
                    "note": "used Fisher due to low expected counts"}
        return {"test": "Chi-square", "stat": round(chi2, 3), "p": round(p_chi, 4),
                "dof": dof, "note": ""}
    else:
        chi2, p_chi, dof, expected = chi2_contingency(arr)
        low = (expected < 5).sum()
        note = f"{low} cells with expected < 5" if low else ""
        return {"test": "Chi-square", "stat": round(chi2, 3), "p": round(p_chi, 4),
                "dof": dof, "note": note}


def sig_stars(p: float) -> str:
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "ns"


# ─────────────────────────────────────────────────────────────────────────────
# TEST 1: Response Language Distribution
# ─────────────────────────────────────────────────────────────────────────────

def test1_response_language(df: pd.DataFrame) -> pd.DataFrame:
    """
    Chi-square / Fisher's exact: cue language × response language.
    Done (a) aggregated across all pairs, and (b) per translation pair.
    Hypothesis (RHM): Hindi cues → more Hindi responses than English cues do.
    """
    print("=" * 70)
    print("TEST 1: Response Language Distribution")
    print("=" * 70)

    # --- 1a. Aggregated ---
    ct_all = pd.crosstab(df["cue_language"], df["response_language"])
    print("\n[1a] Aggregated contingency table (cue_language × response_language):")
    print(ct_all.to_string())
    result_all = run_chi2_or_fisher(ct_all, "aggregated")
    print(f"  → {result_all['test']}: stat={result_all['stat']}, "
          f"p={result_all['p']:.4f} {sig_stars(result_all['p'])}")
    if result_all['note']:
        print(f"     Note: {result_all['note']}")

    # Proportion of Hindi responses per cue language
    prop = df.groupby("cue_language")["response_language"].apply(
        lambda s: (s == "hindi").mean()).rename("prop_hindi_response")
    print(f"\n  Proportion responding in Hindi:")
    print(f"  {prop.to_string()}")

    # --- 1b. Per pair ---
    rows = []
    for eng, hin in PAIRS:
        sub = df[df["pair"] == eng].copy()
        ct = pd.crosstab(sub["cue_language"], sub["response_language"])
        if ct.shape[0] < 2:
            continue
        # Reindex to ensure both languages present
        ct = ct.reindex(index=["english","hindi"], columns=["english","hindi","mixed"],
                        fill_value=0)
        # Use just english/hindi columns for 2×2 Fisher
        ct2 = ct[["english","hindi"]]
        res = run_chi2_or_fisher(ct2, eng)
        # Proportions
        for lang in ["english","hindi"]:
            sub_lang = sub[sub["cue_language"] == lang]
            n = len(sub_lang)
            n_hindi_resp = (sub_lang["response_language"] == "hindi").sum()
            prop_val = n_hindi_resp / n if n > 0 else np.nan
            rows.append({
                "pair": eng,
                "cue_language": lang,
                "n": n,
                "n_hindi_response": n_hindi_resp,
                "prop_hindi_response": round(prop_val, 3),
                "test": res["test"],
                "p": round(res["p"], 4),
                "sig": sig_stars(res["p"]),
                "note": res.get("note",""),
            })

    results_df = pd.DataFrame(rows)
    print("\n[1b] Per-pair results:")
    print(results_df.to_string(index=False))
    results_df.to_csv("/kaggle/working/results/test1_response_language.csv", index=False)

    # --- Figure ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    # Plot 1: stacked bar per pair per cue language
    pivot = results_df.pivot(index="pair", columns="cue_language",
                              values="prop_hindi_response")
    pivot.plot(kind="bar", ax=axes[0], color=["#5B8DB8","#C97F4A"],
               edgecolor="white", width=0.6)
    axes[0].set_title("Proportion of Hindi responses by cue language", fontsize=12)
    axes[0].set_ylabel("Proportion responding in Hindi")
    axes[0].set_xlabel("")
    axes[0].set_ylim(0, 1)
    axes[0].tick_params(axis="x", rotation=45)
    axes[0].axhline(0.5, ls="--", color="gray", lw=0.8, alpha=0.6)

    # Plot 2: aggregated heatmap
    ct_pct = ct_all.div(ct_all.sum(axis=1), axis=0)
    sns.heatmap(ct_pct, annot=True, fmt=".2f", ax=axes[1],
                cmap="Blues", linewidths=0.5, cbar_kws={"label": "Proportion"})
    axes[1].set_title("Response language distribution (row-normalised)", fontsize=12)
    axes[1].set_xlabel("Response language")
    axes[1].set_ylabel("Cue language")

    plt.tight_layout()
    plt.savefig("/kaggle/working/results/figures/test1_response_language.png", dpi=150)
    plt.close()
    print("\n✓ Saved: results/test1_response_language.csv")
    print("✓ Saved: results/figures/test1_response_language.png\n")
    return results_df


# ─────────────────────────────────────────────────────────────────────────────
# TEST 2: Cross-Language Response Overlap (Jaccard Similarity)
# ─────────────────────────────────────────────────────────────────────────────

def test2_response_overlap(df: pd.DataFrame) -> pd.DataFrame:
    """
    For each translation pair compute Jaccard similarity of response sets
    between English and Hindi cue conditions.
    Jaccard = |A ∩ B| / |A ∪ B|  (0=no overlap, 1=identical sets)
    We normalise responses to lowercase to handle capitalisation.
    """
    print("=" * 70)
    print("TEST 2: Cross-Language Response Overlap (Jaccard Similarity)")
    print("=" * 70)

    rows = []
    for eng, hin in PAIRS:
        cat = CATEGORY_MAP[eng]
        sub = df[df["pair"] == eng]
        eng_resp = set(sub[sub["cue_language"] == "english"]["response"]
                       .str.lower().str.strip().dropna())
        hin_resp = set(sub[sub["cue_language"] == "hindi"]["response"]
                       .str.lower().str.strip().dropna())
        intersection = eng_resp & hin_resp
        union        = eng_resp | hin_resp
        jaccard_sim  = len(intersection) / len(union) if union else 0
        rows.append({
            "pair":              eng,
            "category":          cat,
            "n_english_unique":  len(eng_resp),
            "n_hindi_unique":    len(hin_resp),
            "n_shared":          len(intersection),
            "n_union":           len(union),
            "jaccard":           round(jaccard_sim, 3),
            "shared_responses":  ", ".join(sorted(intersection)) or "—",
        })

    results_df = pd.DataFrame(rows).sort_values("jaccard")
    print(results_df[["pair","category","n_english_unique","n_hindi_unique",
                       "n_shared","jaccard","shared_responses"]].to_string(index=False))

    # Overall mean by semantic category
    print("\nMean Jaccard by semantic category:")
    print(results_df.groupby("category")["jaccard"].mean().round(3).to_string())

    results_df.to_csv("/kaggle/working/results/test2_response_overlap.csv", index=False)

    # --- Figure ---
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = {"concrete": "#5B8DB8", "abstract": "#C97F4A",
              "emotional": "#6DAB7B", "cultural": "#B85B7D"}
    bar_colors = [colors[c] for c in results_df["category"]]
    bars = ax.barh(results_df["pair"], results_df["jaccard"],
                   color=bar_colors, edgecolor="white", height=0.6)
    ax.set_xlabel("Jaccard similarity (0 = no overlap, 1 = identical response sets)")
    ax.set_title("Cross-language response set overlap per translation pair", fontsize=12)
    ax.axvline(0.2, ls="--", color="gray", lw=0.8, alpha=0.5)
    # Annotate n_shared
    for bar, row in zip(bars, results_df.itertuples()):
        ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                f"shared={row.n_shared}", va="center", fontsize=8)
    # Legend
    patches = [mpatches.Patch(color=v, label=k) for k, v in colors.items()]
    ax.legend(handles=patches, title="Category", loc="lower right", fontsize=9)
    plt.tight_layout()
    plt.savefig("/kaggle/working/results/figures/test2_response_overlap.png", dpi=150)
    plt.close()
    print("\n✓ Saved: results/test2_response_overlap.csv")
    print("✓ Saved: results/figures/test2_response_overlap.png\n")
    return results_df


# ─────────────────────────────────────────────────────────────────────────────
# TEST 3: Response Type Distribution (Syntagmatic vs Paradigmatic vs Encyclopedic)
# ─────────────────────────────────────────────────────────────────────────────

def test3_response_type(df: pd.DataFrame) -> pd.DataFrame:
    """
    Chi-square test comparing response_type distributions between
    Hindi and English cue conditions, aggregated and per pair.
    response_type column must be coded as: syntagmatic / paradigmatic / encyclopedic
    """
    print("=" * 70)
    print("TEST 3: Response Type Distribution")
    print("=" * 70)

    # --- 3a. Aggregated ---
    ct_all = pd.crosstab(df["cue_language"], df["response_type"])
    print("\n[3a] Aggregated (cue_language × response_type):")
    print(ct_all.to_string())
    res_all = run_chi2_or_fisher(ct_all, "aggregated")
    print(f"  → {res_all['test']}: stat={res_all['stat']}, "
          f"p={res_all['p']:.4f} {sig_stars(res_all['p'])}")

    # Effect size (Cramér's V)
    n = ct_all.values.sum()
    phi2 = res_all["stat"] / n if res_all["stat"] else np.nan
    k = min(ct_all.shape) - 1
    cramers_v = np.sqrt(phi2 / k) if phi2 else np.nan
    print(f"  Cramér's V = {cramers_v:.3f}" if not np.isnan(cramers_v) else "  Cramér's V = n/a")

    # --- 3b. Per pair ---
    rows = []
    for eng, hin in PAIRS:
        sub = df[df["pair"] == eng]
        ct = pd.crosstab(sub["cue_language"], sub["response_type"])
        if ct.shape[0] < 2 or ct.shape[1] < 2:
            continue
        res = run_chi2_or_fisher(ct, eng)
        # Column proportions
        ct_norm = ct.div(ct.sum(axis=1), axis=0)
        for cue_lang in ct_norm.index:
            for rtype in ct_norm.columns:
                rows.append({
                    "pair": eng,
                    "category": CATEGORY_MAP[eng],
                    "cue_language": cue_lang,
                    "response_type": rtype,
                    "proportion": round(ct_norm.loc[cue_lang, rtype], 3),
                    "count": int(ct.loc[cue_lang, rtype]) if cue_lang in ct.index and rtype in ct.columns else 0,
                    "p": round(res["p"], 4),
                    "sig": sig_stars(res["p"]),
                })

    results_df = pd.DataFrame(rows)
    print("\n[3b] Per-pair proportions:")
    print(results_df.to_string(index=False))
    results_df.to_csv("/kaggle/working/results/test3_response_type.csv", index=False)

    # --- Figure: grouped bar, syntagmatic/paradigmatic/encyclopedic by cue language ---
    fig, ax = plt.subplots(figsize=(12, 5))
    plot_df = (results_df.groupby(["response_type","cue_language"])["proportion"]
               .mean().unstack("cue_language").fillna(0))
    plot_df.plot(kind="bar", ax=ax, color=["#5B8DB8","#C97F4A"],
                 edgecolor="white", width=0.6)
    ax.set_title("Mean response type proportion by cue language", fontsize=12)
    ax.set_ylabel("Mean proportion")
    ax.set_xlabel("Response type")
    ax.tick_params(axis="x", rotation=0)
    plt.tight_layout()
    plt.savefig("/kaggle/working/results/figures/test3_response_type.png", dpi=150)
    plt.close()
    print("\n✓ Saved: results/test3_response_type.csv")
    print("✓ Saved: results/figures/test3_response_type.png\n")
    return results_df


# ─────────────────────────────────────────────────────────────────────────────
# TEST 4: Semantic Category Distribution
# ─────────────────────────────────────────────────────────────────────────────

def test4_semantic_category(df: pd.DataFrame) -> pd.DataFrame:
    """
    Chi-square / Fisher's exact comparing semantic_category of responses
    between Hindi and English cue conditions, per translation pair and aggregated.
    semantic_category column coded as: functional / perceptual / emotional /
                                       cultural / taxonomic
    """
    print("=" * 70)
    print("TEST 4: Semantic Category Distribution")
    print("=" * 70)

    ct_all = pd.crosstab(df["cue_language"], df["semantic_category"])
    print("\n[4a] Aggregated (cue_language × semantic_category):")
    print(ct_all.to_string())
    res_all = run_chi2_or_fisher(ct_all, "aggregated")
    print(f"  → {res_all['test']}: stat={res_all['stat']}, "
          f"p={res_all['p']:.4f} {sig_stars(res_all['p'])}")
    if res_all['note']:
        print(f"     Note: {res_all['note']}")

    rows = []
    for eng, hin in PAIRS:
        sub = df[df["pair"] == eng]
        ct = pd.crosstab(sub["cue_language"], sub["semantic_category"])
        if ct.shape[0] < 2:
            continue
        res = run_chi2_or_fisher(ct, eng)
        rows.append({
            "pair":     eng,
            "category": CATEGORY_MAP[eng],
            "test":     res["test"],
            "p":        round(res["p"], 4),
            "sig":      sig_stars(res["p"]),
            "note":     res.get("note",""),
        })

    results_df = pd.DataFrame(rows)
    print("\n[4b] Per-pair significance:")
    print(results_df.to_string(index=False))
    results_df.to_csv("/kaggle/working/results/test4_semantic_category.csv", index=False)

    # --- Figure: heatmap of semantic category proportions per pair × cue language ---
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=False)
    for i, lang in enumerate(["english","hindi"]):
        sub = df[df["cue_language"] == lang]
        ct = pd.crosstab(sub["pair"], sub["semantic_category"])
        ct_pct = ct.div(ct.sum(axis=1), axis=0).fillna(0)
        sns.heatmap(ct_pct, annot=True, fmt=".2f", ax=axes[i],
                    cmap="YlOrRd", vmin=0, vmax=1,
                    linewidths=0.4, cbar_kws={"label": "Proportion"})
        axes[i].set_title(f"{lang.capitalize()} cues – semantic category proportions",
                          fontsize=11)
        axes[i].set_xlabel("Semantic category")
        axes[i].set_ylabel("Word pair")
    plt.tight_layout()
    plt.savefig("/kaggle/working/results/figures/test4_semantic_category.png", dpi=150)
    plt.close()
    print("\n✓ Saved: results/test4_semantic_category.csv")
    print("✓ Saved: results/figures/test4_semantic_category.png\n")
    return results_df


# ─────────────────────────────────────────────────────────────────────────────
# TEST 5: Code-Switching Rate (Asymmetry)
# ─────────────────────────────────────────────────────────────────────────────

def test5_code_switching(df: pd.DataFrame) -> pd.DataFrame:
    """
    A "switch" occurs when the response language differs from the cue language.
    (mixed responses are counted as switches.)
    Compute switch rates for Hindi→English and English→Hindi directions.
    Test asymmetry with a two-sided binomial test (null: switch rate is equal
    in both directions, i.e. p=0.5 when comparing a pooled switch event).
    Also run per-pair.
    """
    print("=" * 70)
    print("TEST 5: Code-Switching Rate and Asymmetry")
    print("=" * 70)

    df = df.copy()
    # A switch = response lang != cue lang (mixed treated as switch)
    df["is_switch"] = df["cue_language"] != df["response_language"]

    # Aggregate switch rates
    switch_rates = df.groupby("cue_language")["is_switch"].agg(["sum","count","mean"])
    switch_rates.columns = ["n_switches","n_total","switch_rate"]
    switch_rates["switch_rate"] = switch_rates["switch_rate"].round(3)
    print("\nAggregate switch rates:")
    print(switch_rates.to_string())

    # Binomial test on Hindi→English switches
    # H0: the probability of switching from Hindi cue is 0.5 (same as English cue)
    hindi_switches  = int(switch_rates.loc["hindi","n_switches"])
    hindi_total     = int(switch_rates.loc["hindi","n_total"])
    english_switches= int(switch_rates.loc["english","n_switches"])
    english_total   = int(switch_rates.loc["english","n_total"])

    # Null: switch rate = pooled switch rate across both conditions
    pooled_rate = (hindi_switches + english_switches) / (hindi_total + english_total)
    binom_hindi   = binomtest(hindi_switches,   hindi_total,   pooled_rate, alternative="two-sided")
    binom_english = binomtest(english_switches, english_total, pooled_rate, alternative="two-sided")

    print(f"\nBinomial test (null: switch prob = pooled rate {pooled_rate:.3f}):")
    print(f"  Hindi cues:   k={hindi_switches}, n={hindi_total}, "
          f"p={binom_hindi.pvalue:.4f} {sig_stars(binom_hindi.pvalue)}")
    print(f"  English cues: k={english_switches}, n={english_total}, "
          f"p={binom_english.pvalue:.4f} {sig_stars(binom_english.pvalue)}")

    # RHM directionality test: does Hindi→English switching > English→Hindi?
    # Compare proportions with chi-square on 2x2 table [switched, not-switched] x [cue_lang]
    ct = pd.DataFrame({
        "switch":     [hindi_switches, english_switches],
        "no_switch":  [hindi_total - hindi_switches, english_total - english_switches],
    }, index=["hindi_cue","english_cue"])
    res = run_chi2_or_fisher(ct, "asymmetry")
    print(f"\nChi-square / Fisher asymmetry test:")
    print(f"  {res['test']}: stat={res['stat']}, p={res['p']:.4f} {sig_stars(res['p'])}")

    # Direction of asymmetry
    hi_rate = switch_rates.loc["hindi","switch_rate"]
    en_rate = switch_rates.loc["english","switch_rate"]
    direction = "Hindi→English switching > English→Hindi" if hi_rate > en_rate else \
                "English→Hindi switching > Hindi→English"
    print(f"  Observed direction: {direction}")
    print(f"  (Hindi cue switch rate={hi_rate:.3f}, English cue switch rate={en_rate:.3f})")

    # Per-pair
    rows = []
    for eng, hin in PAIRS:
        sub = df[df["pair"] == eng]
        for lang in ["english","hindi"]:
            sl = sub[sub["cue_language"] == lang]
            n_sw = sl["is_switch"].sum()
            n_t  = len(sl)
            rows.append({
                "pair":          eng,
                "cue_language":  lang,
                "n_switches":    int(n_sw),
                "n_total":       n_t,
                "switch_rate":   round(n_sw/n_t, 3) if n_t else np.nan,
            })

    results_df = pd.DataFrame(rows)
    print("\nPer-pair switch rates:")
    print(results_df.to_string(index=False))
    results_df.to_csv("/kaggle/working/results/test5_code_switching.csv", index=False)

    # --- Figure ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    # Grouped bar per pair
    pivot = results_df.pivot(index="pair", columns="cue_language",
                              values="switch_rate").fillna(0)
    pivot.plot(kind="bar", ax=axes[0], color=["#5B8DB8","#C97F4A"],
               edgecolor="white", width=0.6)
    axes[0].set_title("Code-switching rate by pair and cue language", fontsize=12)
    axes[0].set_ylabel("Switch rate")
    axes[0].set_ylim(0, 1)
    axes[0].axhline(0.5, ls="--", color="gray", lw=0.8, alpha=0.5)
    axes[0].tick_params(axis="x", rotation=45)
    # Overall bar
    axes[1].bar(["Hindi cue\n→ English resp", "English cue\n→ Hindi resp"],
                [hi_rate, en_rate],
                color=["#C97F4A","#5B8DB8"], edgecolor="white", width=0.5)
    axes[1].set_title("Aggregate directional switch rates", fontsize=12)
    axes[1].set_ylabel("Switch rate")
    axes[1].set_ylim(0, 1)
    axes[1].axhline(pooled_rate, ls="--", color="gray", lw=0.8, alpha=0.5,
                    label=f"Pooled rate ({pooled_rate:.2f})")
    axes[1].legend(fontsize=9)
    plt.tight_layout()
    plt.savefig("/kaggle/working/results/figures/test5_code_switching.png", dpi=150)
    plt.close()
    print("\n✓ Saved: results/test5_code_switching.csv")
    print("✓ Saved: results/figures/test5_code_switching.png\n")
    return results_df


# ─────────────────────────────────────────────────────────────────────────────
# TEST 6: Cultural vs. Concrete Divergence (Type-Token Ratio + Diversity)
# ─────────────────────────────────────────────────────────────────────────────

def test6_cultural_concrete_divergence(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compare response diversity between semantic word categories (concrete,
    abstract, emotional, cultural) and between cue languages.

    Metrics:
      (a) Type-Token Ratio (TTR) = unique responses / total responses
          Higher TTR → more diverse / idiosyncratic associations
      (b) Modal response dominance = frequency of most common response / n
          Lower dominance → more diverse
      (c) Chi-square on TTR differences across categories (Fisher on 2x2 subsets)

    We also compute the cross-language divergence per pair:
      divergence = 1 - Jaccard  (reuses Test 2 logic)
    and correlate it with cue category.
    """
    print("=" * 70)
    print("TEST 6: Cultural vs. Concrete Divergence (Response Diversity)")
    print("=" * 70)

    rows = []
    for eng, hin in PAIRS:
        word_cat = CATEGORY_MAP[eng]  # category of the English member
        for lang in ["english","hindi"]:
            cue = eng if lang == "english" else hin
            cue_cat = CATEGORY_MAP[cue]
            sub = df[(df["cue"] == cue)]
            responses = sub["response"].str.lower().str.strip().dropna()
            n = len(responses)
            if n == 0:
                continue
            n_unique     = responses.nunique()
            ttr          = n_unique / n
            modal_resp   = responses.value_counts().iloc[0] if n_unique else 0
            modal_dom    = modal_resp / n
            top_response = responses.value_counts().index[0] if n_unique else "—"
            rows.append({
                "pair":           eng,
                "cue":            cue,
                "word_category":  word_cat,
                "cue_language":   lang,
                "n":              n,
                "n_unique":       n_unique,
                "ttr":            round(ttr, 3),
                "modal_dominance":round(modal_dom, 3),
                "top_response":   top_response,
            })

    results_df = pd.DataFrame(rows)

    print("\nType-Token Ratio and Modal Dominance by cue word category:")
    summary = results_df.groupby("word_category")[["ttr","modal_dominance"]].mean().round(3)
    print(summary.to_string())

    print("\nTTR by cue language:")
    print(results_df.groupby("cue_language")[["ttr","modal_dominance"]].mean().round(3).to_string())

    # Statistical test: compare TTR across word categories
    # Build a 2×2 table per category pair (unique vs repeated tokens)
    print("\nChi-square: unique tokens vs repeated tokens across cue language × category")
    cat_results = []
    for cat in results_df["word_category"].unique():
        sub = results_df[results_df["word_category"] == cat]
        # unique tokens vs repeated tokens per language
        ct = pd.DataFrame({
            "unique":   sub.groupby("cue_language")["n_unique"].sum(),
            "repeated": (sub.groupby("cue_language")["n"].sum()
                         - sub.groupby("cue_language")["n_unique"].sum()),
        })
        if ct.shape[0] < 2:
            continue
        res = run_chi2_or_fisher(ct, cat)
        cat_results.append({
            "category": cat,
            "mean_ttr_english": round(sub[sub["cue_language"]=="english"]["ttr"].mean(), 3),
            "mean_ttr_hindi":   round(sub[sub["cue_language"]=="hindi"]["ttr"].mean(), 3),
            "test": res["test"],
            "p":    round(res["p"], 4),
            "sig":  sig_stars(res["p"]),
        })

    cat_df = pd.DataFrame(cat_results)
    print(cat_df.to_string(index=False))

    results_df.to_csv("/kaggle/working/results/test6_divergence.csv", index=False)
    cat_df.to_csv("/kaggle/working/results/test6_category_ttr_tests.csv", index=False)

    # --- Figure ---
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    # TTR by word_category × cue_language
    pivot_ttr = results_df.pivot_table(index="word_category", columns="cue_language",
                                        values="ttr", aggfunc="mean").fillna(0)
    pivot_ttr.plot(kind="bar", ax=axes[0], color=["#5B8DB8","#C97F4A"],
                   edgecolor="white", width=0.6)
    axes[0].set_title("Mean TTR by word category and cue language", fontsize=12)
    axes[0].set_ylabel("Type-Token Ratio (higher = more diverse)")
    axes[0].tick_params(axis="x", rotation=0)

    # Modal dominance scatter
    axes[1].scatter(results_df[results_df["cue_language"]=="english"]["ttr"],
                    results_df[results_df["cue_language"]=="english"]["modal_dominance"],
                    label="English cue", color="#5B8DB8", s=60, alpha=0.8)
    axes[1].scatter(results_df[results_df["cue_language"]=="hindi"]["ttr"],
                    results_df[results_df["cue_language"]=="hindi"]["modal_dominance"],
                    label="Hindi cue", color="#C97F4A", s=60, alpha=0.8, marker="^")
    for _, row in results_df.iterrows():
        axes[1].annotate(row["cue"], (row["ttr"], row["modal_dominance"]),
                         fontsize=7, alpha=0.7)
    axes[1].set_xlabel("TTR")
    axes[1].set_ylabel("Modal dominance (higher = one response dominates)")
    axes[1].set_title("TTR vs. modal dominance per cue word", fontsize=12)
    axes[1].legend(fontsize=9)
    plt.tight_layout()
    plt.savefig("/kaggle/working/results/figures/test6_divergence.png", dpi=150)
    plt.close()
    print("\n✓ Saved: results/test6_divergence.csv")
    print("✓ Saved: results/test6_category_ttr_tests.csv")
    print("✓ Saved: results/figures/test6_divergence.png\n")
    return results_df


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main():
    print("\n" + "="*70)
    print("  RQ1 Analysis Pipeline – Hindi-English Cross-Lingual Word Association")
    print("="*70 + "\n")

    df = load_data("/kaggle/input/datasets/navshri2558/responses/responses.csv")

    t1 = test1_response_language(df)
    t2 = test2_response_overlap(df)
    t3 = test3_response_type(df)
    t4 = test4_semantic_category(df)
    t5 = test5_code_switching(df)
    t6 = test6_cultural_concrete_divergence(df)

    print("="*70)
    print("  ALL TESTS COMPLETE")
    print("  Results saved to: results/")
    print("  Figures saved to: results/figures/")
    print("="*70)


if __name__ == "__main__":
    main()


  RQ1 Analysis Pipeline – Hindi-English Cross-Lingual Word Association

✓ Loaded 440 responses from 22 participants, 20 cue words.
  Cue language breakdown:
cue_language
hindi      220
english    220

TEST 1: Response Language Distribution

[1a] Aggregated contingency table (cue_language × response_language):
response_language  english  hindi  mixed
cue_language                            
english                200     20      0
hindi                  129     89      2
  → Chi-square: stat=61.001, p=0.0000 ***
     Note: 2 cells with expected < 5

  Proportion responding in Hindi:
  cue_language
english    0.090909
hindi      0.404545

[1b] Per-pair results:
         pair cue_language  n  n_hindi_response  prop_hindi_response           test      p sig                                   note
      Glasses      english 22                 3                0.136 Fisher's exact 1.0000  ns used Fisher due to low expected counts
      Glasses        hindi 22                 2                